In [ ]:
#single year

def year_hist(year, files, UNIX_year_start, colour, bins = 180):
    chain = ROOT.TChain("events")

    total_time = 0
    jan1_UNIX = UNIX_year_start[year]
    for file in files:
        chain.Add(file)
        
        f = ROOT.TFile(file)
        t = f.Get("events")
        
        tmin = t.GetMinimum("UNIX")
        tmax = t.GetMaximum("UNIX")

        total_time += (tmax - tmin)

        f.Close()
    year_seconds = seconds_in_year(year)

    fraction = total_time/year_seconds

    hist = ROOT.TH1D(f"h_{year}", f"{year};Time since Jan 1 (s);Scaled counts", bins, 0, year_seconds)
    
    chain.Draw(f"(UNIX - {jan1_UNIX}) >> h_{year}", "", "goff")
    if hist.Integral() != 0:
        hist.Scale(fraction / hist.Integral())
    hist.SetLineWidth(2)
    hist.SetLineColor(colour)

    return hist

In [ ]:
#using year_hist

h2023 = year_hist("2023", UoB_year_files["2023"], UNIX_year_start, ROOT.kRed)

h2017 = year_hist("2017", UoB_year_files["2017"], UNIX_year_start, ROOT.kGreen+2)
#h2020 = year_hist("2020", UoB_year_files["2020"], UNIX_year_start, ROOT.kAzure+2)

ymax = 1.1 * max(h2017.GetMaximum(), h2023.GetMaximum())
h2023.SetMaximum(ymax)
h2023.SetMinimum(0)

c = ROOT.TCanvas("c", "", 800, 600)
h2023.SetTitle("2017 vs 2023 University of Birmingham Data")
h2023.Draw("HIST")
#h2020.Draw("HIST SAME")
h2017.Draw("HIST SAME")

leg = ROOT.TLegend(0.75, 0.75, 0.9, 0.9)
leg.AddEntry(h2017, "2017", "l")
#leg.AddEntry(h2020, "2020", "l")
leg.AddEntry(h2023, "2023", "l")
leg.Draw()

c.Draw()


In [ ]:
#year histograms overlayed
def year_hist(year, files, UNIX_year_start, colour, bins = 180):
    chain = ROOT.TChain("events")

    total_time = 0
    jan1_UNIX = UNIX_year_start[year]
    for file in files:
        chain.Add(file)
        
        f = ROOT.TFile(file)
        t = f.Get("events")
        
        tmin = t.GetMinimum("UNIX")
        tmax = t.GetMaximum("UNIX")

        total_time += (tmax - tmin)

        f.Close()
    
    year_seconds = seconds_in_year(year)

    fraction = total_time/year_seconds

    hist = ROOT.TH1D(f"h_{year}", f"{year};Seconds since Jan 1;Scaled counts", bins, 0, year_seconds)
    
    chain.Draw(f"(UNIX - {jan1_UNIX}) >> h_{year}", "", "goff")
    if hist.Integral() != 0:
        hist.Scale(fraction / hist.Integral())
    hist.SetLineWidth(2)
    hist.SetLineColor(colour)

    return hist

In [ ]:
#UoB using year_hist looped

histograms = []
bins = 180

for i,year in enumerate(UoB_year_files):
    colour = colours[i]
    h = year_hist(year, UoB_year_files[year], UNIX_year_start, colour, bins)
    histograms.append((year, h))

ymax = 1.2 * max(h.GetMaximum() for year, h in histograms)
c_all = ROOT.TCanvas("c_all_years", "All Years", 900, 600)

histograms[0][1].SetTitle("University of Birmingham Overlayed Data")

first = True
for year, h in histograms:
    h.SetMaximum(ymax)
    h.SetMinimum(0)

    if first:
        h.Draw("P")
        first = False
    else:
        h.Draw("P SAME")

leg = ROOT.TLegend(0.1, 0.82, 0.9, 0.9)
leg.SetNColumns(len(histograms))

for year, h in histograms:
    leg.AddEntry(h, year, "l")

leg.Draw()

In [ ]:
#def year_hist_alongside

def year_hist_alongside(year, files, UNIX_year_start, colour, offset, total_span, bins=180):

    chain = ROOT.TChain("events")
    total_time = 0.0

    for file in files:
        chain.Add(file)

        f = ROOT.TFile(file)
        t = f.Get("events")

        tmin = t.GetMinimum("UNIX")
        tmax = t.GetMaximum("UNIX")

        total_time += (tmax - tmin)
        f.Close()

    year_seconds = seconds_in_year(year)
    fraction = total_time / year_seconds


    hist = ROOT.TH1D(f"h_{year}", f";Time (years placed side-by-side);Scaled counts", bins, 0, total_span)

    jan1_UNIX = UNIX_year_start[year]
    chain.Draw(f"(UNIX - {jan1_UNIX} + {offset}) >> h_{year}", "", "goff")

    if hist.Integral() != 0:
        hist.Scale(fraction / hist.Integral())

    hist.SetLineWidth(2)
    hist.SetLineColor(colour)
    
    return hist

In [ ]:
#UoB using year_hist_alongside

offset = 0
histograms = []

for i, year in enumerate(UoB_year_files):
    colour = colours[i]
    hist = year_hist_alongside(year, UoB_year_files[year], UNIX_year_start, colour, offset, UoB_total_span)
    histograms.append((year, hist))
    offset += UoB_year_lengths[year]

ymax = 1.2 * max(h.GetMaximum() for _, h in histograms)


c_side = ROOT.TCanvas("c_side_by_side", "Years Side By Side", 1100, 500)

histograms[0][1].SetTitle("University of Birmingham Data")

first = True
for year, h in histograms:
    h.SetMaximum(ymax)
    h.SetMinimum(0)

    if first:
        h.Draw("HIST")
        first = False
    else:
        h.Draw("HIST SAME")


leg = ROOT.TLegend(0.1, 0.82, 0.9, 0.9)
leg.SetNColumns(len(histograms))

for year, h in histograms:
    leg.AddEntry(h, year, "l")

leg.Draw()

c_side.Draw()